In [ ]:
# ============================================================
# 0. MOUNT GOOGLE DRIVE
# ============================================================

#from google.colab import drive

#drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import shutil
import cv2
import yaml
import torch

#!pip install -q ultralytics
from ultralytics import YOLO

In [38]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# VIDEO_NAME = "Sample_BioReactor_video_cam1.mov" 
VIDEO_NAME = "20260831_111348_f.mp4"

CVAT_NAME = "CVAT_2026_09_24_test_02"

RUN_BASELINE = False

from pathlib import Path

DRIVE_ROOT = Path(".") 

VIDEO_FOLDER = DRIVE_ROOT / "Data_Videos_1_9-16-2026"

VIDEO = VIDEO_FOLDER / VIDEO_NAME
# VIDEO = DRIVE_ROOT / VIDEO_NAME

CVAT_DIR = DRIVE_ROOT / CVAT_NAME
CVAT_YAML = CVAT_DIR / "data.yaml"
CVAT_LABEL_DIR = CVAT_DIR / "labels" / "train"

WORK_DIR = Path("./CAPcell_YOLO")
DATASET_DIR = WORK_DIR / "dataset"
IMAGE_DIR = DATASET_DIR / "images" / "train"
LABEL_DIR = DATASET_DIR / "labels" / "train"

RUNS_DIR = DRIVE_ROOT / f"runs_{CVAT_NAME}"

BASE_MODEL = "yolo26n.pt"

EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 8
CONFIDENCE = 0.20

In [22]:
# # ============================================================
# # 2. CONFIGURATION
# # ============================================================


# # Hi Emmaluz and Mateo: You only need to modify, at most, the following:
# #
# # 1. The video from which we take the samples:
# #
# # VIDEO_NAME = "Sample_BioReactor_video_cam1.mov"
# #VIDEO_NAME = "/content/drive/Shareddrives/ERISE_capcell/Data_Videos_1_9-16-2026/20260831_111348_f.mp4"
# #
# #
# # 2. The original CVAT export. Please follow this naming convention:
# #    CVAT_2026_09_17_test_01
# #    Increase the test number for additional experiments on the same day.
# #
# # CVAT_NAME = "CVAT_2026_09_23_test_02"
# #
# # 3. If you change the video or the YOLO base model, we need to run
# #    the baseline model again. Otherwise, the baseline is the same
# #    as in the first test:
# #
# # RUN_BASELINE = False
# #
# # Don't change anything else! Hahaha

# # ------------------------------------------------------------
# # Shared Drive
# # ------------------------------------------------------------
# DRIVE_ROOT = Path(
#    "/content/drive/Shareddrives/ERISE_capcell"
# )
# # ------------------------------------------------------------
# # Original video
# # ------------------------------------------------------------
# VIDEO = DRIVE_ROOT / VIDEO_NAME

# # ------------------------------------------------------------
# # CVAT paths
# # ------------------------------------------------------------
# CVAT_DIR = DRIVE_ROOT / CVAT_NAME
# CVAT_YAML = CVAT_DIR / "data.yaml"
# CVAT_LABEL_DIR = CVAT_DIR / "labels" / "train"

# # ------------------------------------------------------------
# # Temporary dataset used for training
# #
# # This is created locally in Colab.
# # ------------------------------------------------------------
# WORK_DIR = Path(
#     "/content/CAPcell_YOLO"
# )

# DATASET_DIR = WORK_DIR / "dataset"
# IMAGE_DIR = DATASET_DIR / "images" / "train"
# LABEL_DIR = DATASET_DIR / "labels" / "train"

# # ------------------------------------------------------------
# # Training outputs
# # ------------------------------------------------------------
# RUNS_DIR = DRIVE_ROOT / f"runs_{CVAT_NAME}"

# # ------------------------------------------------------------
# # YOLO model
# # ------------------------------------------------------------
# BASE_MODEL = "yolo26n.pt"

# # ------------------------------------------------------------
# # Training parameters
# # ------------------------------------------------------------
# EPOCHS = 50
# IMAGE_SIZE = 640
# BATCH_SIZE = 8

# CONFIDENCE = 0.20

In [39]:
# ============================================================
# 3. CHECK ENVIRONMENT
# ============================================================

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "WARNING: No GPU detected"
    )

ENVIRONMENT
PyTorch: 2.14.0+cpu
CUDA available: False


In [40]:
# ============================================================
# 4. CHECK INPUT FILES
# ============================================================

print("\n" + "=" * 60)
print("INPUT FILES")
print("=" * 60)


if not DRIVE_ROOT.exists():

    raise FileNotFoundError(
        f"Shared Drive folder not found:\n{DRIVE_ROOT}"
    )


if not VIDEO.exists():

    raise FileNotFoundError(
        f"Video not found:\n{VIDEO}"
    )


if not CVAT_DIR.exists():

    raise FileNotFoundError(
        f"CVAT dataset not found:\n{CVAT_DIR}"
    )


if not CVAT_YAML.exists():

    raise FileNotFoundError(
        f"CVAT data.yaml not found:\n{CVAT_YAML}"
    )


if not CVAT_LABEL_DIR.exists():

    raise FileNotFoundError(
        f"CVAT labels not found:\n{CVAT_LABEL_DIR}"
    )


print("✓ Shared Drive found")
print("✓ Video found")
print("✓ CVAT dataset found")
print("✓ data.yaml found")
print("✓ Labels found")


INPUT FILES
✓ Shared Drive found
✓ Video found
✓ CVAT dataset found
✓ data.yaml found
✓ Labels found


In [41]:
# ============================================================
# 5. READ CVAT data.yaml
# ============================================================

print("\n" + "=" * 60)
print("READING CVAT DATASET CONFIGURATION")
print("=" * 60)


with open(CVAT_YAML, "r") as f:

    cvat_data = yaml.safe_load(f)

# ------------------------------------------------------------
# Extract class names
# ------------------------------------------------------------

names = cvat_data.get("names")

if names is None:

    raise ValueError(
        "No 'names' field found in CVAT data.yaml."
    )

# CVAT/YOLO may store names as either:
#
# {0: particle, 1: other}
#
# or:
#
# [particle, other]
#
# We normalize both cases.

if isinstance(names, dict):

    class_names = {
        int(k): str(v)
        for k, v in names.items()
    }

elif isinstance(names, list):

    class_names = {
        i: str(name)
        for i, name in enumerate(names)
    }

else:

    raise ValueError(
        "Unsupported format for 'names' in data.yaml."
    )


print("\nClasses:")

for class_id, class_name in class_names.items():

    print(
        f"  {class_id}: {class_name}"
    )


NUM_CLASSES = len(class_names)

print(
    f"\nNumber of classes: {NUM_CLASSES}"
)


READING CVAT DATASET CONFIGURATION

Classes:
  0: particle 1
  1: particle 2
  2: particle 3

Number of classes: 3


In [42]:
# ============================================================
# 6. IDENTIFY ANNOTATED FRAMES AUTOMATICALLY
# ============================================================

print("\n" + "=" * 60)
print("IDENTIFYING ANNOTATED FRAMES")
print("=" * 60)


label_files = sorted(
    CVAT_LABEL_DIR.glob("frame_*.txt")
)


if not label_files:

    raise FileNotFoundError(
        "No frame label files found."
    )


annotated_frames = []


for label_file in label_files:

    # Example:
    #
    # frame_000024.txt
    #
    # → 24

    frame_id = int(
        label_file.stem.split("_")[-1]
    )

    annotated_frames.append(
        frame_id
    )


annotated_frames = sorted(
    set(annotated_frames)
)


print(
    f"Annotated frames found: "
    f"{len(annotated_frames)}"
)

print(
    f"First frame: "
    f"{annotated_frames[0]}"
)

print(
    f"Last frame: "
    f"{annotated_frames[-1]}"
)


IDENTIFYING ANNOTATED FRAMES
Annotated frames found: 64
First frame: 0
Last frame: 63


In [43]:
# ============================================================
# 7. PREPARE LOCAL DATASET
# ============================================================

print("\n" + "=" * 60)
print("PREPARING LOCAL DATASET")
print("=" * 60)


# Remove previous temporary dataset

if DATASET_DIR.exists():

    print(
        "Removing previous temporary dataset..."
    )

    shutil.rmtree(
        DATASET_DIR
    )


IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LABEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PREPARING LOCAL DATASET
Removing previous temporary dataset...


In [44]:
# ============================================================
# 8. EXTRACT ANNOTATED FRAMES FROM VIDEO
# ============================================================

print("\n" + "=" * 60)
print("EXTRACTING VIDEO FRAMES")
print("=" * 60)


cap = cv2.VideoCapture(str(VIDEO))

if not cap.isOpened():
    raise RuntimeError(
        f"Could not open video:\n{VIDEO}"
    )


video_frame_count = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

print(
    f"Video frames available: {video_frame_count}"
)

print(
    f"Frames requested: {len(annotated_frames)}"
)


# Convert to set for fast lookup
annotated_set = set(annotated_frames)

extracted = 0
current_frame = 0


# ------------------------------------------------------------
# Read the video sequentially
# ------------------------------------------------------------

while True:

    success, frame = cap.read()

    if not success:
        break


    # Save only frames that have annotations
    if current_frame in annotated_set:

        output_file = (
            IMAGE_DIR
            / f"frame_{current_frame:06d}.png"
        )

        success_write = cv2.imwrite(
            str(output_file),
            frame
        )

        if not success_write:

            raise RuntimeError(
                f"Could not save frame "
                f"{current_frame}"
            )

        extracted += 1


    current_frame += 1


cap.release()


print(
    f"\nFrames read from video: {current_frame}"
)

print(
    f"Frames extracted: {extracted}"
)


# ------------------------------------------------------------
# Verify that every annotated frame was extracted
# ------------------------------------------------------------

extracted_frames = sorted(
    int(f.stem.split("_")[-1])
    for f in IMAGE_DIR.glob("frame_*.png")
)


missing = sorted(
    annotated_set - set(extracted_frames)
)


if missing:

    print(
        "\nWARNING: The following annotated "
        "frames could not be extracted:"
    )

    print(missing)

else:

    print(
        "\n✓ All annotated frames were extracted"
    )


EXTRACTING VIDEO FRAMES
Video frames available: 1501
Frames requested: 64

Frames read from video: 1501
Frames extracted: 64

✓ All annotated frames were extracted


In [45]:
# ============================================================
# 9. COPY CVAT LABELS
# ============================================================

print("\n" + "=" * 60)
print("COPYING CVAT LABELS")
print("=" * 60)


copied = 0


for frame_id in annotated_frames:

    source = (
        CVAT_LABEL_DIR
        / f"frame_{frame_id:06d}.txt"
    )


    destination = (
        LABEL_DIR
        / f"frame_{frame_id:06d}.txt"
    )


    if source.exists():

        shutil.copy2(
            source,
            destination
        )

        copied += 1


print(
    f"✓ Copied {copied} label files"
)


COPYING CVAT LABELS
✓ Copied 64 label files


In [46]:
# ============================================================
# 10. CREATE TRAINING YAML
# ============================================================

print("\n" + "=" * 60)
print("CREATING YOLO DATASET YAML")
print("=" * 60)


DATA_YAML = (
    DATASET_DIR
    / "data.yaml"
)


# ------------------------------------------------------------
# IMPORTANT:
#
# We do NOT modify CVAT's original data.yaml.
#
# This YAML is generated specifically for
# the local Colab dataset.
# ------------------------------------------------------------


training_yaml = {

    "path": str(DATASET_DIR),

    "train": "images/train",

    # For this preliminary experiment only.
    # Later we will create a real validation split.
    "val": "images/train",

    "names": class_names
}


with open(DATA_YAML, "w") as f:

    yaml.safe_dump(
        training_yaml,
        f,
        sort_keys=False
    )


print(
    DATA_YAML.read_text()
)


CREATING YOLO DATASET YAML
path: CAPcell_YOLO\dataset
train: images/train
val: images/train
names:
  0: particle 1
  1: particle 2
  2: particle 3



In [47]:
# ============================================================
# 11. VERIFY DATASET
# ============================================================

print("\n" + "=" * 60)
print("VERIFYING DATASET")
print("=" * 60)


images = sorted(
    IMAGE_DIR.glob("*.png")
)

labels = sorted(
    LABEL_DIR.glob("*.txt")
)


print(
    f"Images: {len(images)}"
)

print(
    f"Labels: {len(labels)}"
)


# ------------------------------------------------------------
# Check image-label correspondence
# ------------------------------------------------------------

image_ids = {
    f.stem
    for f in images
}

label_ids = {
    f.stem
    for f in labels
}


missing_images = (
    label_ids
    - image_ids
)

missing_labels = (
    image_ids
    - label_ids
)


if missing_images:

    raise RuntimeError(
        "Missing images for labels:\n"
        + "\n".join(
            sorted(missing_images)
        )
    )


if missing_labels:

    raise RuntimeError(
        "Missing labels for images:\n"
        + "\n".join(
            sorted(missing_labels)
        )
    )


print(
    "✓ Every image has a corresponding label"
)


VERIFYING DATASET
Images: 64
Labels: 64
✓ Every image has a corresponding label


In [48]:
# ============================================================
# 12. CHECK LABEL CONTENT AND CLASS IDS
# ============================================================

print("\n" + "=" * 60)
print("CHECKING LABELS")
print("=" * 60)


class_counts = {
    class_id: 0
    for class_id in class_names
}


total_annotations = 0


for label_file in labels:

    with open(
        label_file,
        "r"
    ) as f:

        lines = [
            line.strip()
            for line in f
            if line.strip()
        ]


    for line in lines:

        values = line.split()

        if len(values) != 5:

            raise ValueError(
                f"Invalid YOLO label in "
                f"{label_file.name}:\n"
                f"{line}"
            )


        class_id = int(
            values[0]
        )


        if class_id not in class_names:

            raise ValueError(
                f"Class ID {class_id} found in "
                f"{label_file.name}, but it is "
                f"not defined in data.yaml."
            )


        class_counts[class_id] += 1

        total_annotations += 1


print(
    f"Total annotations: "
    f"{total_annotations}"
)


print("\nClass distribution:")


for class_id, count in class_counts.items():

    print(
        f"  {class_id}: "
        f"{class_names[class_id]} "
        f"→ {count}"
    )


print(
    "\n✓ Label format and class IDs are valid"
)





CHECKING LABELS
Total annotations: 335

Class distribution:
  0: particle 1 → 0
  1: particle 2 → 254
  2: particle 3 → 81

✓ Label format and class IDs are valid


In [49]:
# ============================================================
# 13. BASELINE YOLO
# ============================================================

print("\n" + "=" * 60)
print("STEP 1 - BASELINE YOLO")
print("=" * 60)


if RUN_BASELINE:

    baseline_model = YOLO(BASE_MODEL)

    baseline_model.predict(
        source=str(VIDEO),
        conf=CONFIDENCE,
        imgsz=IMAGE_SIZE,
        save=True,
        project=str(RUNS_DIR),
        name="baseline",
        exist_ok=True,
        device=0
    )

else:
    print("Baseline skipped.")

print(
    "\n✓ Baseline detection completed"
)


print(
    "Output:"
)

print(
    RUNS_DIR / "baseline"
)





STEP 1 - BASELINE YOLO
Baseline skipped.

✓ Baseline detection completed
Output:
runs_CVAT_2026_09_24_test_02\baseline


In [50]:
# ============================================================
# 14. FINE-TUNING
# ============================================================

print("\n" + "=" * 60)
print("STEP 2 - YOLO FINE-TUNING")
print("=" * 60)


print(
    f"Model      : {BASE_MODEL}"
)

print(
    f"Dataset    : {DATA_YAML}"
)

print(
    f"Classes    : {NUM_CLASSES}"
)

print(
    f"Images     : {len(images)}"
)

print(
    f"Annotations: {total_annotations}"
)

print(
    f"Epochs     : {EPOCHS}"
)

print(
    f"Image size : {IMAGE_SIZE}"
)

print(
    f"Batch size : {BATCH_SIZE}"
)


model_ft = YOLO(
    BASE_MODEL
)


training_results = model_ft.train(

    data=str(DATA_YAML),

    epochs=EPOCHS,

    imgsz=IMAGE_SIZE,

    batch=BATCH_SIZE,

    project=str(RUNS_DIR),

    name="capcell_finetune",

    exist_ok=True,

    device="cpu"
    # device=0
)


print(
    "\n✓ Fine-tuning completed"
)


STEP 2 - YOLO FINE-TUNING
Model      : yolo26n.pt
Dataset    : CAPcell_YOLO\dataset\data.yaml
Classes    : 3
Images     : 64
Annotations: 335
Epochs     : 50
Image size : 640
Batch size : 8
Ultralytics 8.4.162  Python-3.14.7 torch-2.14.0+cpu CPU (Intel Core Ultra 7 265)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=CAPcell_YOLO\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=N

In [51]:
# ============================================================
# 15. LOCATE BEST MODEL
# ============================================================

# BEST_MODEL = (

#     RUNS_DIR

#     / "capcell_finetune"

#     / "weights"

#     / "best.pt"
# )

BEST_MODEL = (
    Path("runs")
    / "detect"
    / f"runs_{CVAT_NAME}"
    / "capcell_finetune"
    / "weights"
    / "best.pt"
)


if not BEST_MODEL.exists():

    raise FileNotFoundError(
        f"best.pt not found:\n"
        f"{BEST_MODEL}"
    )


print(
    "\nBest model:"
)

print(
    BEST_MODEL
)


# ============================================================
# 16. RUN FINE-TUNED MODEL ON ORIGINAL VIDEO
# ============================================================

print("\n" + "=" * 60)
print("STEP 3 - FINE-TUNED YOLO ON ORIGINAL VIDEO")
print("=" * 60)


finetuned_model = YOLO(
    str(BEST_MODEL)
)


finetuned_model.predict(

    source=str(VIDEO),

    conf=CONFIDENCE,

    imgsz=IMAGE_SIZE,

    save=True,

    project=str(RUNS_DIR),

    name="finetuned_video",

    exist_ok=True,

    device="cpu"
    # device=0
)


print(
    "\n✓ Fine-tuned detection completed"
)


print(
    "\nOutput:"
)

print(
    RUNS_DIR / "finetuned_video"
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("EXPERIMENT COMPLETED")
print("=" * 60)


print(
    "\nInput video:"
)

print(
    VIDEO
)


print(
    "\nOriginal CVAT dataset:"
)

print(
    CVAT_DIR
)


print(
    "\nTemporary training dataset:"
)

print(
    DATASET_DIR
)


print(
    "\nNumber of classes:"
)

print(
    NUM_CLASSES
)


print(
    "\nNumber of training images:"
)

print(
    len(images)
)


print(
    "\nNumber of annotations:"
)

print(
    total_annotations
)


print(
    "\nBaseline:"
)

print(
    RUNS_DIR / "baseline"
)


print(
    "\nFine-tuned model:"
)

print(
    BEST_MODEL
)


print(
    "\nFine-tuned video:"
)

print(
    RUNS_DIR / "finetuned_video"
)


print(
    "\nDone."
)


Best model:
runs\detect\runs_CVAT_2026_09_24_test_02\capcell_finetune\weights\best.pt

STEP 3 - FINE-TUNED YOLO ON ORIGINAL VIDEO

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1501) c:\Users\MateoFlorez\Desktop\ERISE\Data_Videos_1_9-16-2026\20260831_111348_f.mp4: 384x640 (no detections), 25.4ms
video 1/1 (frame 2/1501) c:\Users\MateoFlorez\Desktop\ERISE\Data_Videos_1_9-16-2026\20260831_111348_f.mp4: 384x640 (no detections), 15.4ms
video 1/1 (frame 3/1501) c:\Users\MateoFlorez\Desktop